# Gerar o APK no Google Colab

Este notebook prepara o ambiente Linux, compila o projeto com Buildozer e baixa o APK de depuraÃ§Ã£o. O primeiro build pode levar bastante tempo porque baixa SDK, NDK e dependÃªncias.

Use o arquivo Projeto_My_Music_Gospel_Colab.zip gerado junto com este notebook. Ele contÃ©m o cÃ³digo atualizado, banco, imagens e fontes, sem MP3.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()  # Selecione Projeto_My_Music_Gospel_Colab.zip
zip_name = next(iter(uploaded))
project_dir = Path("/content/app_musica")
project_dir.mkdir(parents=True, exist_ok=True)
with ZipFile(zip_name) as archive:
    archive.extractall(project_dir)
assert (project_dir / "buildozer.spec").is_file(), "O ZIP nÃ£o contÃ©m buildozer.spec na raiz"
print("Projeto pronto em:", project_dir)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq git zip unzip curl openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev automake autopoint gettext
!curl https://sh.rustup.rs -sSf | sh -s -- -y

In [ ]:
%cd /content/app_musica
import os
from pathlib import Path
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
!python -m pip install --upgrade pip setuptools wheel buildozer cython==0.29.34
!buildozer --version
!java -version

## Compilar

O buildozer.spec jÃ¡ configura o pacote Android, permissÃµes, API 35, NDK 28c e o serviÃ§o Java de reproduÃ§Ã£o. O SDK e o NDK serÃ£o baixados pelo Buildozer na primeira execuÃ§Ã£o.

In [ ]:
!buildozer -v android debug

In [ ]:
from pathlib import Path
from google.colab import files

apks = list(Path("/content/app_musica/bin").glob("*.apk"))
assert apks, "O build terminou sem gerar APK. Consulte a saÃ­da acima para localizar o erro."
apk = max(apks, key=lambda item: item.stat().st_mtime)
print("APK:", apk)
files.download(str(apk))

### ObservaÃ§Ãµes

- Este APK Ã© de depuraÃ§Ã£o e pode ser instalado para testar no celular.
- As mÃºsicas nÃ£o sÃ£o incluÃ­das no APK; o app as procura no armazenamento do aparelho, conforme descrito no README.
- Se a sessÃ£o do Colab for encerrada, serÃ¡ necessÃ¡rio compilar novamente.